## Subsample list of extracted URLs from wikipedia articles for using as high-quality data

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from fastwarc.warc import ArchiveIterator, WarcRecordType
import resiliparse.extract.html2text
from typing import Literal
import rich


In [3]:
from cs336_data import data_utils 

In [4]:
s = "hello world this is an english sentence"
data_utils.identify_language(s), data_utils.is_english(s)

(('en', 0.8897625803947449), True)

In [5]:
WIKI_DATA_PATH = "/scratch/ac6074/cs336-a4-data/wiki/subsampled_positive_urls_10k.warc.warc.gz"
CC_EXAMPLE_PATH = "/scratch/ac6074/cs336-a4-data/CC/example.warc.gz"
FASTTEXT_TRAIN_FILE = "/scratch/ac6074/cs336-a4-data/fasttext_train_quality.txt"

In [ ]:
upto = 70000
total_good_count = 0
for i, (entry, text) in enumerate(data_utils.warc_text_iterator(CC_EXAMPLE_PATH)):
    if i > upto: 
        break
    classified_text = data_utils.simple_pipeline(text)
    if classified_text not in ["NOT_ENGLISH", "NSFW", "HATESPEECH", "BAD_GOPHER"]:
        total_good_count += 1
        print(total_good_count)
        # print(f"{i}\n{classified_text}\n\n")



In [7]:
upto = 10000
for i, (entry, text) in enumerate(data_utils.warc_text_iterator(WIKI_DATA_PATH)):
    if i > upto: 
        break
    print(f"Ind={i}\n{text}\n\n---------------\n\n")
    classified_text = data_utils.simple_pipeline(text)
    if classified_text not in ["NOT_ENGLISH", "NSFW", "HATESPEECH", "BAD_GOPHER"]:
        print(f"{i}\n{classified_text}\n\n")

Ind=0
software: Wget/1.24.5 (linux-gnu) format: WARC File Format 1.0 conformsTo: http://bibnum.bnf.fr/WARC/WARC_ISO_28500_version1_latestdraft.pdf robots: classic wget-arguments: "--timeout=5" "-i" "subsampled_positive_urls_10k.txt" "--warc-file=subsampled_positive_urls_10k.warc" "-O" "/dev/null"

---------------


Ind=1
"--timeout=5" "-i" "subsampled_positive_urls_10k.txt" "--warc-file=subsampled_positive_urls_10k.warc" "-O" "/dev/null"

---------------


Ind=2
Opening WARC file ‘subsampled_positive_urls_10k.warc.warc.gz’. --2026-10-20 10:40:39-- https://filmcommission.nl/database/producer/39903/ Resolving filmcommission.nl (filmcommission.nl)... 149.210.160.195 Connecting to filmcommission.nl (filmcommission.nl)|149.210.160.195|:443... connected. HTTP request sent, awaiting response... 301 Moved Permanently Location: https://www.filmcommission.nl/database/producer/39903/ [following] 0K 100% 85.9M=0s --2026-10-20 10:40:40-- https://www.filmcommission.nl/database/producer/39903/ Resolv

In [ ]:
upto = 100
with open(WIKI_DATA_PATH, "rb") as f:
    it = ArchiveIterator(f)
    for i, entry in enumerate(it):
        if i > 100: break
        if entry.headers.get("Content-Type", "NONE") not in ["application/http; msgtype=response", "application/http;msgtype=response"]: continue
        try: 
            warc_str = entry.reader.read().decode()
            warc_txt = resiliparse.extract.html2text.extract_plain_text(warc_str).strip()
            if len(warc_txt) == 0: continue
            print(entry.header)
            # print(f"Ind={i}\n{text}\n\n---------------\n\n")
        except UnicodeDecodeError:
            continue

: 

## Make a file for training fasttext
Each line is of the form
``` 
__label__high [text], OR
__label__low [text]
```
Once we have a file, we can train using 
```
fasttext supervised -input training_data.txt -output my_classifier_model 
```

In [ ]:
upto = 100
with open(FASTTEXT_TRAIN_FILE, "w") as fasttext_train: 
    for i, (entry, text) in enumerate(data_utils.warc_text_iterator(CC_EXAMPLE_PATH)):
        if i > upto: break
        piplined_text = data_utils.simple_pipeline(text)
        if piplined_text is not None:
            fasttext_train.write(f"__label__low {piplined_text}\n")

    for i, (entry, text) in enumerate(data_utils.warc_text_iterator(WIKI_DATA_PATH)):
        if i > upto: break
        piplined_text = data_utils.simple_pipeline(text)
        if piplined_text is not None:
            fasttext_train.write(f"__label__high {piplined_text}\n")
